In [1]:
# Handling survival analysis or right-censored data in Numpyro

import jax
import numpyro
numpyro.set_host_device_count(4)
from jax import numpy as jnp, random

from numpyro.distributions import RightCensoredDistribution, Weibull
from numpyro import distributions as dist

In [2]:
wb = Weibull(1.0, 2.0)

In [3]:
times = jnp.array([0.5, 1.0, 1.5, 2.0])
events = jnp.array([1, 1, 0, 0])  # 1 if event observed, 0 if censored

In [4]:
print(wb.log_prob(times))
print(jnp.log1p(-wb.cdf(times)))

[-0.25       -0.30685282 -1.1513877  -2.6137056 ]
[-0.25      -1.        -2.2499998 -4.0000014]


In [5]:
wbc = RightCensoredDistribution(wb, censored=1 - events)
print(wbc.log_prob(times))

[-0.25       -0.30685282 -2.2499998  -4.0000014 ]


In [6]:
# simulate some data and check parameter retrieval

scale = 1.0
concentration = 2.0
censor_prob = .2
n = 10000

key = random.PRNGKey(0)
key_time, key_censor, key = random.split(key, 3)
wb = Weibull(scale, concentration)
times = wb.sample(key_time, (n,))
is_censored = dist.Bernoulli(censor_prob).sample(key_censor, (n,))

# save to csv
import pandas as pd
df = pd.DataFrame({"times": times, "is_censored": is_censored})
df.to_csv("survival_data.csv", index=False)

In [7]:
from numpyro.infer.hmc import NUTS
from numpyro.infer.mcmc import MCMC

def model(times, is_censored):
    scale = numpyro.sample("scale", dist.HalfNormal(10))
    concentration = numpyro.sample("concentration", dist.HalfNormal(10))
    wb = Weibull(scale, concentration)
    wbc = RightCensoredDistribution(wb, censored=is_censored)
    numpyro.sample("obs", wbc, obs=times)

In [8]:
mcmc = MCMC(NUTS(model), num_warmup=1000, num_samples=2000, num_chains=4)
mcmc.run(key, times, is_censored)
mcmc.print_summary()

  0%|          | 0/3000 [00:00<?, ?it/s]

  0%|          | 0/3000 [00:00<?, ?it/s]

  0%|          | 0/3000 [00:00<?, ?it/s]

  0%|          | 0/3000 [00:00<?, ?it/s]


                     mean       std    median      5.0%     95.0%     n_eff     r_hat
  concentration      1.99      0.02      1.99      1.96      2.01   4615.46      1.00
          scale      1.12      0.01      1.12      1.11      1.13   7521.93      1.00

Number of divergences: 0
